## Shared setup

In [2]:
!pip install facenet-pytorch tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 103.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 121.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 720.4 kB/s eta 0:00:000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 89.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 61.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### force crash the Kernel to restart it and load libraries

In [ ]:
import os
os.kill(os.getpid(), 9) # This forces the kernel to restart automatically

: 

: 

: 

In [1]:
import os
import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw
from pathlib import Path
from tqdm import tqdm
from facenet_pytorch import MTCNN, InceptionResnetV1
from google.colab import drive

In [6]:
# --- Mount Google Drive ---
drive.flush_and_unmount()
drive.mount("/content/drive")

# --- Folder Setup ---
IMAGES_DIR = Path("/content/all_images")                                    # unzipped images
DRIVE_SAVE = Path("/content/drive/MyDrive/walk-of-life/face_recognition")   # .pt (embeddings) and .pkl (metadata) save location
EMBEDDINGS_FILE = DRIVE_SAVE / "embeddings_db.pt" 

DRIVE_SAVE.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


##### Device

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Part 1: extract emdedding for later identification and save results on drive

### load dataset

In [8]:
!cp "/content/drive/MyDrive/walk-of-life/dataset1.zip" "/content/"
!cp "/content/drive/MyDrive/walk-of-life/dataset2.zip" "/content/"
!cp "/content/drive/MyDrive/walk-of-life/dataset3.zip" "/content/"

In [ ]:
!mkdir -p /content/all_images/
!unzip -q /content/dataset1.zip -d /content/all_images/dataset1/
!unzip -q /content/dataset2.zip -d /content/all_images/dataset2/
!unzip -q /content/dataset3.zip -d /content/all_images/dataset3/

### define embedding database

In [11]:
def build_embedding_database(
    images_dir: Path,
    save_path: Path,
    batch_size: int = 16,
    image_size: int = 160,
    min_face_prob: float = 0.90,
):
    """
    Iterate every image in `images_dir`, detect all faces with MTCNN,
    extract 512-d embeddings via InceptionResnetV1, and persist the
    result to `save_path` as a .pt file.

    Saved structure (list of dicts):
      [
        {
          "filename":  str,           # e.g. "IMG_4821.jpg"
          "bbox":      list[float],   # [x1, y1, x2, y2] in original pixels
          "prob":      float,         # MTCNN detection confidence
          "embedding": Tensor(512,),  # L2-normalised, on CPU
        },
        ...
      ]
    """

    # --- Models ---
    # keep_all=True  → return every detected face, not just the best one
    # post_process=True → normalise pixel values to [-1, 1] for the resnet
    mtcnn = MTCNN(
        image_size=image_size,
        keep_all=True,
        post_process=True,
        device=device,
        min_face_size=40,          # ignore very small/blurry faces
        thresholds=[0.6, 0.7, 0.7] # MTCNN stage thresholds (P/R/O-net)
    )

    resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

    # --- Collect image paths ---
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    # Use rglob to search recursively through all subfolders
    image_paths = [
        p for p in sorted(images_dir.rglob("*"))
        if p.is_file() and p.suffix.lower() in valid_ext
    ]
    print(f"Found {len(image_paths):,} images in {images_dir}")

    records = []
    skipped = 0

    for img_path in tqdm(image_paths, desc="Extracting embeddings", unit="img"):
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            tqdm.write(f"  [SKIP] Cannot open {img_path.name}: {e}")
            skipped += 1
            continue

        # --- Detect faces ---
        # boxes  : Tensor[N, 4] or None
        # probs  : Tensor[N]    or None
        # faces  : Tensor[N, 3, H, W] or None   (aligned & cropped)
        try:
            boxes, probs, faces = mtcnn.detect(img, landmarks=False), None, None
            # detect() only returns boxes+probs; forward() returns cropped faces
            faces = mtcnn(img)          # [N, 3, 160, 160] or None
            boxes, probs = mtcnn.detect(img)
        except Exception as e:
            tqdm.write(f"  [SKIP] MTCNN error on {img_path.name}: {e}")
            skipped += 1
            continue

        if faces is None or boxes is None:
            continue  # no faces detected in this image

        # Filter by confidence
        keep = probs >= min_face_prob
        if not keep.any():
            continue

        faces  = faces[keep]     # [K, 3, 160, 160]
        boxes  = boxes[keep]     # [K, 4]
        probs  = probs[keep]     # [K]

        # --- Extract embeddings ---
        with torch.no_grad():
            face_batch = faces.to(device)               # [K, 3, 160, 160]
            embeddings = resnet(face_batch)              # [K, 512]  (L2-normed by default)
            embeddings = embeddings.cpu()

        # --- Store one record per detected face ---
        for i in range(len(embeddings)):
            records.append({
                # Save the subfolder path too (e.g., "dataset1/IMG_001.jpg")
                "filename":  str(img_path.relative_to(images_dir)),
                "bbox":      boxes[i].tolist(),          # [x1, y1, x2, y2]
                "prob":      float(probs[i]),
                "embedding": embeddings[i],              # Tensor(512,)
            })

    print(f"\nDone. {len(records):,} face records from "
          f"{len(image_paths) - skipped:,} images "
          f"({skipped} skipped).")
 
    # --- Save to Drive ---
    torch.save(records, save_path)
    print(f"Saved → {save_path}  ({save_path.stat().st_size / 1e6:.1f} MB)")
    return records


#### run embedding extraction

In [12]:
records = build_embedding_database(IMAGES_DIR, EMBEDDINGS_FILE)

  0%|          | 0.00/107M [00:00<?, ?B/s]

Found 10,554 images in /content/all_images


Extracting embeddings:   4%|▍         | 406/10554 [04:47<1:59:43,  1.41img/s]


KeyboardInterrupt: 

## Part 2: Search


### load database

In [ ]:
def load_database(path: Path):
    """Load the .pt embedding database and stack embeddings into a matrix."""
    records = torch.load(path, map_location="cpu")

    # Stack all embeddings into one matrix for vectorised similarity search
    emb_matrix = torch.stack([r["embedding"] for r in records])  # [N, 512]

    print(f"Loaded {len(records):,} face records from {path}")
    return records, emb_matrix

In [ ]:
records, emb_matrix = load_database(EMBEDDINGS_FILE)

## target

In [ ]:
QUERY_IMAGE = "/content/drive/MyDrive/walk-of-life/face_recognition/mesposito.jpg"   # ← target face

### Extract the target embedding used to query the embedding db

In [ ]:
def get_query_embedding(
    query_image_path: str,
    image_size: int = 160,
) -> torch.Tensor:
    """
    Detect the largest/most-confident face in the query image and return
    its 512-d L2-normalised embedding.

    Raises ValueError if no face is found.
    """
    mtcnn_query = MTCNN(
        image_size=image_size,
        keep_all=False,      # single best face for the query
        post_process=True,
        device=device,
    )
    resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

    img = Image.open(query_image_path).convert("RGB")
    face = mtcnn_query(img)          # Tensor[3, 160, 160] or None

    if face is None:
        raise ValueError(f"No face detected in {query_image_path}. "
                         "Check the image is well-lit and the face is unobstructed.")

    with torch.no_grad():
        embedding = resnet(face.unsqueeze(0).to(device))   # [1, 512]
    return embedding.squeeze(0).cpu()                       # [512]

In [ ]:
query_emb = get_query_embedding(QUERY_IMAGE)
print(f"Query embedding shape: {query_emb.shape}")

### Cosine similarity search

In [ ]:
def search(
    query_emb: torch.Tensor,
    emb_matrix: torch.Tensor,
    records: list,
    top_k: int = 5,
    distance_metric: str = "cosine",   # "cosine" | "euclidean"
) -> list[dict]:
    """
    Find the top-k closest faces to the query embedding.

    Returns a list of dicts, each containing the original record plus:
      "score"     : float   (cosine similarity, higher = better)
      "distance"  : float   (euclidean distance, lower = better)
      "rank"      : int
    """
    q = query_emb.unsqueeze(0)   # [1, 512]

    # Cosine similarity (both sides are already L2-normalised by InceptionResnetV1)
    cos_sim = torch.nn.functional.cosine_similarity(q, emb_matrix, dim=1)  # [N]

    # Euclidean distance (useful as a secondary sanity check)
    eu_dist = torch.cdist(q, emb_matrix, p=2).squeeze(0)                   # [N]

    if distance_metric == "cosine":
        scores, indices = torch.topk(cos_sim, k=top_k, largest=True)
    else:
        scores, indices = torch.topk(eu_dist, k=top_k, largest=False)
        scores = -scores   # invert so "higher = better" is consistent

    results = []
    for rank, (idx, score) in enumerate(zip(indices.tolist(), scores.tolist()), 1):
        entry = dict(records[idx])          # copy
        entry["score"]    = cos_sim[idx].item()
        entry["distance"] = eu_dist[idx].item()
        entry["rank"]     = rank
        results.append(entry)

    return results

### printout

In [ ]:
top_matches = search(query_emb, emb_matrix, records, top_k=30)

for m in top_matches:
    print(f"  #{m['rank']}  {m['filename']:<40}  "
          f"cos={m['score']:.4f}  eu={m['distance']:.4f}")

In [ ]:
def display_matches(
    matches: list[dict],
    images_dir: Path,
    query_image_path: str,
    bbox_color: str = "#FF3B30",
    figsize_per_match: tuple = (4, 4),
):
    """
    Display the query face alongside the top-k matched images, each with a
    coloured bounding box drawn around the matched face.

    Requires the original images to be available locally (images_dir).
    Falls back to showing only the metadata if the image file is missing.
    """
    n = len(matches) + 1          # +1 for the query thumbnail
    fig, axes = plt.subplots(1, n, figsize=(figsize_per_match[0] * n, figsize_per_match[1]))

    # --- Query image (left panel) ---
    ax0 = axes[0]
    try:
        q_img = Image.open(query_image_path).convert("RGB")
        ax0.imshow(q_img)
    except Exception:
        ax0.text(0.5, 0.5, "Query\n(not found)", ha="center", va="center",
                 transform=ax0.transAxes, fontsize=10)
    ax0.set_title("Query face", fontsize=11, fontweight="bold", pad=8)
    ax0.axis("off")

    # --- Match panels ---
    for ax, match in zip(axes[1:], matches):
        img_path = images_dir / match["filename"]

        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)

            # Draw bounding box
            x1, y1, x2, y2 = match["bbox"]
            rect = patches.FancyBboxPatch(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2.5,
                edgecolor=bbox_color,
                facecolor="none",
                boxstyle="round,pad=2",
            )
            ax.add_patch(rect)

            # Confidence label above the bbox
            ax.text(
                x1, y1 - 8,
                f"cos {match['score']:.3f}",
                color="white",
                fontsize=8,
                fontweight="bold",
                bbox=dict(facecolor=bbox_color, edgecolor="none", pad=2, alpha=0.85),
            )

        except FileNotFoundError:
            ax.text(0.5, 0.5,
                    f"{match['filename']}\n(image not local)",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="gray")

        short_name = match["filename"]
        if len(short_name) > 28:
            short_name = "…" + short_name[-25:]

        ax.set_title(
            f"#{match['rank']}  {short_name}\n"
            f"cos={match['score']:.4f}  eu={match['distance']:.4f}",
            fontsize=8, pad=6,
        )
        ax.axis("off")

    plt.suptitle("Top Face Matches", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Display! ---
# images_dir is only needed for Part 2 visualisation.
# If you're in a fresh session without the raw images, pass a path
# that doesn't exist; the function will degrade gracefully.
display_matches(top_matches, IMAGES_DIR, QUERY_IMAGE)